# 50-Agent Swarm for an Approximate Closed-Form American Option Equation

## Research objective

American equity options are a useful test of whether a large language-model swarm can contribute to a problem that is simultaneously mathematical, numerical, economic, and institutional. The obstacle is familiar: unlike the standard European Black–Scholes contract, an American option embeds an optimal stopping decision. The holder may exercise before maturity, so valuation is not merely an expectation of a terminal payoff. The price must respect a free boundary separating continuation from exercise. For general dividends, interest rates, volatility, maturity, and moneyness, there is no single elementary Black–Scholes-style expression that exactly solves the problem. Numerical methods such as binomial trees, finite differences, and least-squares Monte Carlo therefore remain standard reference methods.

This notebook does **not** ask fifty agents to hallucinate a miraculous exact formula. It asks a more disciplined question: can a heterogeneous swarm propose useful mathematical structure for a **compact approximate equation**, while a deterministic numerical layer decides what survives? The distinction is essential. The language models perform hypothesis generation, decomposition, critique, and structural selection. They do not manufacture option prices and they do not decide whether a candidate approximation is accurate. Ground truth is generated independently with a Cox–Ross–Rubinstein American-option tree. Candidate structures are translated into an explicit feature library, estimated on a training sample, and judged on observations the swarm never sees.

The target representation is deliberately interpretable. We write the American price as

$
V_A(S,K,T,r,q,\sigma)=V_E(S,K,T,r,q,\sigma)+\Pi(S,K,T,r,q,\sigma),
$

where $V_E$is the corresponding European Black–Scholes value and $\Pi\ge 0$ is the early-exercise premium. This decomposition gives the swarm a strong economic prior. It also makes the approximation easier to audit: the swarm is searching for a compact representation of the premium rather than rediscovering the entire option-pricing function from scratch. For a non-dividend-paying American call, the notebook imposes the classical no-early-exercise result and sets the premium to zero. For puts and dividend-paying calls, the premium is learned from numerically generated American prices.

The notebook uses **50 Claude Sonnet 5 agents**, arranged as **10 specialist families × 5 variants**. The families are intentionally different. Some agents think like option theorists and focus on free-boundary economics. Others focus on asymptotics, dimensional consistency, symbolic regression, numerical stability, arbitrage restrictions, dividend effects, moneyness regimes, maturity regimes, robustness, or model criticism. The five variants within each family receive slightly different mandates. Heterogeneity therefore comes from differentiated intellectual roles rather than random temperature. Each agent must return structured JSON under a common contract. The contract limits the agent to a predeclared feature vocabulary and requires a short rationale, a proposed feature subset, interaction terms, regime warnings, and explicit failure modes.

The swarm is only one layer of the architecture. A deterministic governance layer validates every response, rejects unknown features, clips excessive complexity, records provenance, and converts accepted proposals into a vote table. Features with broad support across independent specialist families receive more weight than features championed by only one cluster. This is important because fifty correlated opinions are not equivalent to fifty independent discoveries. The notebook therefore measures both raw agent support and family-level support. It also compares subswarms of 5, 10, 20, 30, 40, and 50 agents to test whether the inferred structure stabilizes as the swarm grows.

The numerical experiment is synthetic by design. We sample economically plausible combinations of spot, strike, maturity, interest rate, dividend yield, and volatility. Each contract is valued with a sufficiently deep CRR tree. The corresponding European price and standard Black–Scholes quantities are computed analytically. We then construct dimensionless explanatory variables such as log-moneyness $x=\ln(K/S)$, $\sigma\sqrt{T}$, $rT$, $qT$, and selected nonlinear interactions. Dimensionless variables help the approximation generalize across nominal price scales and make the final expression easier to interpret.

The swarm does not directly estimate coefficients. Instead, its accepted votes determine a compact candidate basis. The deterministic layer fits coefficients using ridge-regularized least squares on the **normalized early-exercise premium**, with safeguards that enforce economically sensible behavior at the prediction stage. A positive-part operator prevents a negative premium. Standard lower bounds—intrinsic value and European value—are imposed when reconstructing the American price. Upper bounds are also enforced: an American call cannot exceed spot, and an American put cannot exceed strike. These projections do not make a poor approximation good, but they prevent obvious arbitrage violations from contaminating evaluation.

The notebook reports several kinds of evidence. First, it measures approximation error against the CRR benchmark using MAE, RMSE, median absolute error, and high-percentile error. Second, it reports errors by option type and by economically important regimes such as short maturity, high volatility, deep in/out of the money, and positive dividends. Third, it exposes the final equation with fitted coefficients so that the result can be copied into another model or paper. Fourth, it maps agent disagreement and family support, preserving the intellectual provenance of the approximation. Fifth, it tests swarm-size convergence to ask a question that is central to collective AI: did additional agents actually add information, or merely add volume?

The resulting equation should be interpreted as an **empirical closed-form approximation**, not as an exact theorem. Its domain is the parameter region represented in the training experiment. A formula that performs well on the synthetic grid may deteriorate under extreme rates, very long maturities, discontinuous dividends, stochastic volatility, jumps, transaction costs, or other departures from the Black–Scholes assumptions. The notebook therefore ends with stress tests and explicit scope conditions.

Pedagogically, the exercise illustrates a broader architecture for advanced autonomous systems. The LLM swarm is used where generative intelligence is valuable: proposing representations, challenging assumptions, and discovering compact structure. Deterministic mathematics is used where reproducibility and verification are essential: pricing the benchmark, estimating coefficients, enforcing constraints, and measuring error. The workflow is therefore not “ask fifty agents for an answer and average them.” It is **hypothesis swarm → governed contracts → numerical evidence → robust synthesis → falsification → equation**. That division of labor is the central methodological lesson of the notebook.

## CELL 1 — Runtime, reproducibility, and governed swarm substrate

This first unit establishes the computational substrate before any agent is allowed to reason about option pricing. That ordering is deliberate. A swarm experiment is only scientifically useful if the environment records the model, random seed, numerical conventions, and response contract. The notebook therefore separates configuration from inference and makes the API key an external secret rather than embedding credentials in code.

The code imports the numerical stack, defines global constants, initializes the Anthropic client from Colab Secrets, and fixes random seeds. It also declares the ten specialist families that will later generate five agents each. The families are not decorative labels: they are the mechanism by which we create controlled heterogeneity. An arbitrage specialist receives a different mandate from an asymptotics specialist, while all agents remain bound to the same output schema.

The feature vocabulary is also declared here. Agents are not permitted to invent arbitrary executable mathematics. They may select only from a transparent library of dimensionless features and interactions that the deterministic layer knows how to compute. This protects the workflow from malformed symbolic output and makes every candidate auditable. The feature set is intentionally broad enough to express nonlinear moneyness, maturity, volatility, rates, dividends, and interactions, but small enough to preserve interpretability.

Finally, the unit defines the JSON response contract. Each agent must identify its family and variant, choose a bounded number of features, nominate interactions, explain its economic rationale, state where it expects the approximation to fail, and provide a confidence score. Later cells validate these fields. The key methodological principle is that the LLM may propose structure, but it cannot silently change the experiment.

In [6]:
# CELL 1 — Runtime, reproducibility, and governed swarm substrate
# Current Anthropic SDK baseline: v1.7.0

!pip -q install "anthropic==1.7.0" scipy scikit-learn

import os, json, re, math, time, random, itertools, warnings
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
from importlib.metadata import version as package_version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import norm
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

from google.colab import userdata
from anthropic import Anthropic


# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 766

rng = np.random.default_rng(SEED)
random.seed(SEED)


# ============================================================
# MODEL AND SWARM CONFIGURATION
# ============================================================

MODEL = "claude-sonnet-5"

N_AGENTS = 50
N_FAMILIES = 10
VARIANTS_PER_FAMILY = 5

TREE_STEPS = 300

RIDGE_ALPHA = 1e-5

MAX_FEATURES_PER_AGENT = 10
MAX_FINAL_FEATURES = 14

ANTHROPIC_SDK_VERSION = package_version("anthropic")


# ============================================================
# ANTHROPIC CLIENT
# ============================================================

API_KEY = userdata.get("ANTHROPIC_API_KEY")

if not API_KEY:
    raise ValueError(
        "Add ANTHROPIC_API_KEY to Colab Secrets before running the swarm."
    )

client = Anthropic(api_key=API_KEY)


# ============================================================
# SPECIALIST FAMILIES
# ============================================================

FAMILIES = {

    "free_boundary":
        "Think like an optimal-stopping and free-boundary theorist.",

    "asymptotics":
        "Think in short-time, long-time, small-premium, and limiting regimes.",

    "arbitrage_bounds":
        "Prioritize monotonicity, bounds, and economically admissible structure.",

    "dividends_carry":
        "Prioritize rates, dividends, carry, and early exercise incentives.",

    "moneyness_geometry":
        "Prioritize log-moneyness and distance from the exercise region.",

    "maturity_regimes":
        "Prioritize maturity scaling and near-expiry behavior.",

    "volatility_structure":
        "Prioritize volatility scaling and volatility-moneyness interactions.",

    "symbolic_regression":
        "Seek a sparse, interpretable symbolic basis with useful interactions.",

    "numerical_stability":
        "Seek stable, low-collinearity terms that extrapolate gracefully.",

    "adversarial_critic":
        "Attack likely misspecifications and nominate terms needed to repair them.",
}


# ============================================================
# GOVERNED FEATURE VOCABULARY
# ============================================================

FEATURE_DESCRIPTIONS = {

    "x":
        "log-moneyness x = ln(K/S)",

    "abs_x":
        "|x|",

    "x2":
        "x^2",

    "sqrtT":
        "sqrt(T)",

    "T":
        "T",

    "sig_sqrtT":
        "sigma*sqrt(T)",

    "varT":
        "sigma^2*T",

    "rT":
        "r*T",

    "qT":
        "q*T",

    "carryT":
        "(r-q)*T",

    "disc_r":
        "exp(-r*T)",

    "disc_q":
        "exp(-q*T)",

    "x_sig":
        "x*sigma*sqrt(T)",

    "absx_sig":
        "|x|*sigma*sqrt(T)",

    "x_rT":
        "x*r*T",

    "x_qT":
        "x*q*T",

    "sig_rT":
        "sigma*sqrt(T)*r*T",

    "sig_qT":
        "sigma*sqrt(T)*q*T",

    "sqrtT_r":
        "sqrt(T)*r",

    "sqrtT_q":
        "sqrt(T)*q",

    "put_itm":
        "max(x,0)",

    "call_itm":
        "max(-x,0)",

    "put_itm2":
        "max(x,0)^2",

    "call_itm2":
        "max(-x,0)^2",
}


ALLOWED_FEATURES = list(FEATURE_DESCRIPTIONS)


# ============================================================
# RUNTIME REPORT
# ============================================================

print("=" * 72)
print("AMERICAN OPTION — 50 AGENT RESEARCH SWARM")
print("=" * 72)

print(f"Model:             {MODEL}")
print(f"Anthropic SDK:     {ANTHROPIC_SDK_VERSION}")
print(
    f"Architecture:      "
    f"{N_FAMILIES} families × "
    f"{VARIANTS_PER_FAMILY} variants "
    f"= {N_AGENTS} agents"
)
print(f"Allowed features:  {len(ALLOWED_FEATURES)}")
print(f"Random seed:       {SEED}")
print(f"CRR tree steps:    {TREE_STEPS}")

print("=" * 72)

AMERICAN OPTION — 50 AGENT RESEARCH SWARM
Model:             claude-sonnet-5
Anthropic SDK:     1.7.0
Architecture:      10 families × 5 variants = 50 agents
Allowed features:  24
Random seed:       766
CRR tree steps:    300


## CELL 2 — Independent numerical ground truth: European analytics and American CRR tree

A swarm searching for an approximation needs an external judge. In this notebook that judge is a Cox–Ross–Rubinstein binomial tree, not another language model. This unit implements the Black–Scholes European call and put equations and a vector-friendly CRR American pricer with early exercise at every node. The numerical benchmark embodies the optimal stopping feature that makes the American contract difficult.

The European value is not merely a baseline for comparison. It is the anchor of the final approximation. We model the early-exercise premium rather than the total price, because the European component is already known in closed form. This reduces the statistical burden and incorporates strong financial structure into the learned equation. The decomposition also makes a crucial exact case easy to enforce: in the standard continuous-dividend Black–Scholes setting, a non-dividend-paying American call should not be exercised early, so its value equals the European call.

The tree uses risk-neutral up and down factors and discounts backward from maturity. At each node the continuation value is compared with immediate intrinsic value, and the larger quantity is retained. The implementation includes basic parameter checks and a configurable number of time steps. Increasing the number of steps improves the benchmark at the cost of computation; the notebook uses a moderate default for dataset generation and allows a deeper tree for spot checks.

This cell is intentionally deterministic and contains no LLM call. That separation prevents circular validation: the agents will later reason about mathematical form, but they cannot grade their own proposals.

In [7]:
# CELL 2 — Independent numerical ground truth: European analytics and American CRR tree
def bs_price(S, K, T, r, q, sigma, option_type):
    if T <= 0:
        return max(S-K, 0.0) if option_type == "call" else max(K-S, 0.0)
    vol = sigma * math.sqrt(T)
    d1 = (math.log(S/K) + (r-q+0.5*sigma*sigma)*T) / vol
    d2 = d1 - vol
    if option_type == "call":
        return S*math.exp(-q*T)*norm.cdf(d1) - K*math.exp(-r*T)*norm.cdf(d2)
    return K*math.exp(-r*T)*norm.cdf(-d2) - S*math.exp(-q*T)*norm.cdf(-d1)

def american_crr(S, K, T, r, q, sigma, option_type, steps=TREE_STEPS):
    if T <= 0:
        return max(S-K, 0.0) if option_type == "call" else max(K-S, 0.0)
    dt = T / steps
    u = math.exp(sigma * math.sqrt(dt))
    d = 1.0 / u
    growth = math.exp((r-q)*dt)
    p = (growth-d) / (u-d)
    p = min(max(p, 0.0), 1.0)
    disc = math.exp(-r*dt)

    j = np.arange(steps + 1)
    ST = S * (u ** j) * (d ** (steps-j))
    values = np.maximum(ST-K, 0.0) if option_type == "call" else np.maximum(K-ST, 0.0)

    for n in range(steps-1, -1, -1):
        values = disc * (p*values[1:n+2] + (1-p)*values[:n+1])
        j = np.arange(n + 1)
        Sn = S * (u ** j) * (d ** (n-j))
        intrinsic = np.maximum(Sn-K, 0.0) if option_type == "call" else np.maximum(K-Sn, 0.0)
        values = np.maximum(values, intrinsic)
    return float(values[0])

# Sanity checks
tests = [
    (100,100,1.0,0.04,0.00,0.20,"call"),
    (100,100,1.0,0.04,0.00,0.20,"put"),
    (100,100,1.0,0.04,0.05,0.20,"call"),
]
for args in tests:
    eu = bs_price(*args)
    am = american_crr(*args)
    print(args[-1], "European=", round(eu,4), "American=", round(am,4), "Premium=", round(am-eu,4))

call European= 9.9251 American= 9.9184 Premium= -0.0066
put European= 6.004 American= 6.4013 Premium= 0.3973
call European= 7.1466 American= 7.3016 Premium= 0.1549


## CELL 3 — Synthetic option universe and dimensionless state variables

The third unit creates the laboratory in which candidate equations will be tested. Rather than selecting a handful of familiar contracts, we generate a broad synthetic universe spanning moneyness, maturity, rates, dividend yields, and volatility. This is important because an approximation can look excellent near at-the-money conditions while failing badly in the tails or near expiry.

Spot is normalized around 100 and strike is generated through log-moneyness. This gives direct control over economically meaningful relative states rather than arbitrary nominal prices. Maturities include short and medium horizons; volatility ranges from quiet to stressed equity conditions; rates and continuous dividend yields cover zero and positive values. Both calls and puts are included. The dataset is then valued independently with the CRR tree and with European Black–Scholes.

From those prices we compute the early-exercise premium and normalize it by strike. The feature library is dimensionless: log-moneyness, powers and absolute values of moneyness, square-root maturity, total variance scale, rate-time, dividend-time, discount factors, and interactions. This choice reflects dimensional analysis. A useful closed-form approximation should not change simply because the same economic contract is quoted in cents rather than dollars.

The sample is divided into training and holdout sets with a fixed seed. The agents never receive holdout outcomes. In fact, they do not need to see raw rows at all; they receive only the variable definitions, domain ranges, and research objective. Their task is structural hypothesis generation. Numerical fitting and evaluation remain downstream. This creates a clean boundary between generative reasoning and empirical verification.

In [8]:
# CELL 3 — Synthetic option universe and dimensionless state variables
def feature_frame(df):
    z = pd.DataFrame(index=df.index)
    x = np.log(df["K"] / df["S"])
    sqrtT = np.sqrt(df["T"])
    sigsqrt = df["sigma"] * sqrtT
    z["x"] = x
    z["abs_x"] = np.abs(x)
    z["x2"] = x*x
    z["sqrtT"] = sqrtT
    z["T"] = df["T"]
    z["sig_sqrtT"] = sigsqrt
    z["varT"] = df["sigma"]**2 * df["T"]
    z["rT"] = df["r"] * df["T"]
    z["qT"] = df["q"] * df["T"]
    z["carryT"] = (df["r"]-df["q"]) * df["T"]
    z["disc_r"] = np.exp(-df["r"]*df["T"])
    z["disc_q"] = np.exp(-df["q"]*df["T"])
    z["x_sig"] = x*sigsqrt
    z["absx_sig"] = np.abs(x)*sigsqrt
    z["x_rT"] = x*df["r"]*df["T"]
    z["x_qT"] = x*df["q"]*df["T"]
    z["sig_rT"] = sigsqrt*df["r"]*df["T"]
    z["sig_qT"] = sigsqrt*df["q"]*df["T"]
    z["sqrtT_r"] = sqrtT*df["r"]
    z["sqrtT_q"] = sqrtT*df["q"]
    z["put_itm"] = np.maximum(x,0)
    z["call_itm"] = np.maximum(-x,0)
    z["put_itm2"] = np.maximum(x,0)**2
    z["call_itm2"] = np.maximum(-x,0)**2
    return z

def make_universe(n=2200, seed=SEED, stress=False):
    rg = np.random.default_rng(seed)
    if stress:
        x = rg.uniform(-0.55, 0.55, n)
        T = np.exp(rg.uniform(np.log(1/365), np.log(5.0), n))
        sigma = rg.uniform(0.08, 0.90, n)
        r = rg.uniform(0.0, 0.12, n)
        q = rg.uniform(0.0, 0.12, n)
    else:
        x = rg.uniform(-0.35, 0.35, n)
        T = np.exp(rg.uniform(np.log(7/365), np.log(3.0), n))
        sigma = rg.uniform(0.10, 0.60, n)
        r = rg.uniform(0.0, 0.08, n)
        q = rg.uniform(0.0, 0.08, n)
    S = np.full(n, 100.0)
    K = S*np.exp(x)
    typ = rg.choice(["call","put"], size=n)
    return pd.DataFrame({"S":S,"K":K,"T":T,"r":r,"q":q,"sigma":sigma,"type":typ})

df = make_universe()
df["euro"] = [bs_price(*row) for row in df[["S","K","T","r","q","sigma","type"]].itertuples(index=False, name=None)]
df["american"] = [american_crr(*row) for row in df[["S","K","T","r","q","sigma","type"]].itertuples(index=False, name=None)]
df["premium"] = np.maximum(df["american"] - df["euro"], 0.0)
df["premium_norm"] = df["premium"] / df["K"]

perm = rng.permutation(len(df))
cut = int(0.75*len(df))
train_idx, test_idx = perm[:cut], perm[cut:]
train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

print(df[["american","euro","premium"]].describe())
print("\nTrain:", len(train), "Holdout:", len(test))

           american          euro      premium
count  2.200000e+03  2.200000e+03  2200.000000
mean   1.277222e+01  1.253643e+01     0.236093
std    1.234569e+01  1.209893e+01     0.711120
min    1.118251e-72  5.634766e-59     0.000000
25%    1.181840e+00  1.172404e+00     0.000000
50%    9.331034e+00  9.193545e+00     0.005890
75%    2.201920e+01  2.171345e+01     0.130142
max    6.467104e+01  6.297761e+01     9.446509

Train: 1650 Holdout: 550


## CELL 4 — Instantiate 50 differentiated agents: 10 specialist families × 5 variants

This unit turns the abstract swarm into fifty explicit research roles. Ten families are created: free-boundary theory, asymptotics, arbitrage and bounds, dividends and carry, moneyness geometry, maturity regimes, volatility structure, symbolic regression, numerical stability, and adversarial model criticism. Each family contains five variants whose prompts emphasize different aspects of the same specialty.

The purpose is not to simulate fifty personalities. It is to create a controlled ensemble of mathematical perspectives. A free-boundary agent may emphasize distance to the exercise region; a dividend specialist may prioritize $qT$ and carry interactions; an asymptotics agent may focus on short-time scaling; an adversarial critic may nominate features specifically because simpler specifications are likely to fail without them. Because all agents use the same underlying model, role differentiation is more scientifically defensible than pretending that stochastic sampling alone creates independent expertise.

Every prompt describes the pricing decomposition, the permitted feature vocabulary, the parameter domain, and the requirement for a compact approximation. Agents are explicitly told that no exact elementary solution is presumed and that they must not fabricate benchmark errors. They are asked only to propose structure. Responses are requested as JSON and stored with provenance.

The cell builds the 50 prompts but does not yet call the API. This makes the experiment inspectable before spending tokens: the researcher can print any mandate, verify the family balance, or alter the permitted feature set. In an autonomous-system workflow, this is analogous to reviewing institutional charters before allowing institutions to act.

In [12]:
# CELL 4 — Instantiate 50 differentiated agents:
# 10 specialist families × 5 variants

VARIANT_LENSES = [
    "Favor the smallest defensible basis and explain what can be omitted.",
    "Focus on interactions that capture regime changes without becoming a black box.",
    "Focus on limiting cases and what the formula must do near those limits.",
    "Focus on robustness across the full stated parameter domain.",
    "Act as a skeptical coauthor: propose structure, then identify its most likely failure.",
]


# ============================================================
# RESPONSE DISCIPLINE
# ============================================================
#
# The actual JSON structure will be enforced by Anthropic's
# structured-output schema in CELL 5.
#
# Here we control CONTENT LENGTH so agents do not generate
# unnecessarily large contracts.
# ============================================================

schema_text = """
Your response will be constrained by a JSON schema.

Content requirements:

- features:
  choose 3 to 10 names ONLY from ALLOWED_FEATURES

- must_have_interactions:
  choose 0 to 4 names ONLY from ALLOWED_FEATURES

- rationale:
  maximum 100 words;
  explain the mathematical/economic intuition concisely

- expected_failure_modes:
  maximum 3 items;
  each item should be one short sentence

- confidence:
  number from 0 to 1

Do not provide derivations, essays, markdown, commentary,
benchmark errors, numerical coefficients, or text outside
the required structured response.
"""


domain = """
We seek an empirical closed-form approximation for American
equity options under constant r, q, and sigma.

The structural decomposition is

    V_A = V_E_BS + early_exercise_premium.

Training domain:

    S = 100 normalization
    ln(K/S) in [-0.35, 0.35]
    T approximately 7 days to 3 years
    sigma 10% to 60%
    r 0% to 8%
    q 0% to 8%
    calls and puts

For non-dividend-paying calls, q = 0, the exact theoretical
relationship C_A = C_E is imposed separately.

Your task is STRUCTURAL HYPOTHESIS GENERATION ONLY.

Independent Cox-Ross-Rubinstein numerical data will estimate
coefficients and evaluate accuracy downstream.

Do not claim an exact analytical solution.
Do not invent performance statistics.
Do not estimate coefficients.
"""


# ============================================================
# BUILD THE 50 RESEARCH MANDATES
# ============================================================

prompts = []


for family, mandate in FAMILIES.items():

    for variant, lens in enumerate(
        VARIANT_LENSES,
        start=1
    ):

        prompt = f"""
You are agent {family}-{variant} in a governed
50-agent mathematical research swarm.

SPECIALIST FAMILY
{family}

SPECIALTY
{mandate}

VARIANT LENS
{lens}


RESEARCH PROBLEM
{domain}


PERMITTED FEATURE VOCABULARY

{json.dumps(
    FEATURE_DESCRIPTIONS,
    indent=2
)}


RESEARCH TASK

Select a compact basis for the normalized early-exercise premium

    (V_A - V_E) / K.

Prefer dimensionless and economically interpretable structure.

The deterministic numerical layer—not you—will:

1. estimate coefficients;
2. force the early-exercise premium to be non-negative;
3. impose elementary option-price bounds;
4. evaluate the approximation against CRR prices;
5. test the formula on unseen contracts.

You are therefore contributing a mathematical hypothesis,
not a price prediction.


OUTPUT DISCIPLINE

{schema_text}
"""

        prompts.append(
            {
                "family": family,
                "variant": variant,
                "prompt": prompt,
            }
        )


# ============================================================
# INSPECTION
# ============================================================

print(
    f"Prepared prompts: {len(prompts)}"
)

assert len(prompts) == N_AGENTS

print()
print("=" * 72)
print("EXAMPLE AGENT MANDATE")
print("=" * 72)

print(
    prompts[0]["prompt"][:2500]
)

Prepared prompts: 50

EXAMPLE AGENT MANDATE

You are agent free_boundary-1 in a governed
50-agent mathematical research swarm.

SPECIALIST FAMILY
free_boundary

SPECIALTY
Think like an optimal-stopping and free-boundary theorist.

VARIANT LENS
Favor the smallest defensible basis and explain what can be omitted.


RESEARCH PROBLEM

We seek an empirical closed-form approximation for American
equity options under constant r, q, and sigma.

The structural decomposition is

    V_A = V_E_BS + early_exercise_premium.

Training domain:

    S = 100 normalization
    ln(K/S) in [-0.35, 0.35]
    T approximately 7 days to 3 years
    sigma 10% to 60%
    r 0% to 8%
    q 0% to 8%
    calls and puts

For non-dividend-paying calls, q = 0, the exact theoretical
relationship C_A = C_E is imposed separately.

Your task is STRUCTURAL HYPOTHESIS GENERATION ONLY.

Independent Cox-Ross-Rubinstein numerical data will estimate
coefficients and evaluate accuracy downstream.

Do not claim an exact analytica

**What Cell 4 Is Really Doing**

Cell 4 is the point where the notebook stops being just a pricing experiment and becomes a genuine multi-agent research system.

Up to this point, the notebook has already created a large artificial universe of American options and has calculated reference prices using conventional numerical methods. Cell 4 does something very different. It does not price anything. It does not estimate coefficients. It does not decide whether a proposed formula is good or bad.

Instead, Cell 4 creates fifty different research “points of view.”

The easiest way to understand it is to imagine that we have organized a research conference. We have invited fifty researchers and divided them into ten departments. Each department specializes in a different way of thinking about the same problem. Within every department, five researchers receive slightly different instructions.

All fifty are looking at the same broad question:

How can we describe the extra value of an American option, relative to an otherwise comparable European option, using a relatively small and interpretable set of variables?

That is the entire intellectual purpose of Cell 4.

The agents are not being asked to discover the final answer. They are being asked to suggest what the final answer might need to pay attention to.

This distinction is very important.

An American option gives its owner an additional right that a European option does not have: the possibility of exercising before the final expiration date. That extra flexibility can have economic value. The notebook calls that difference the “early-exercise premium.”

Cell 4 asks the agents:

“What kinds of market conditions should matter when we try to describe that extra value?”

The agents answer by selecting candidate variables and explaining why they think those variables matter.

The notebook will later determine, using actual numerical data, which suggestions were useful.

So the agents propose hypotheses. The numerical system later tests them.

That division of labor is one of the most interesting aspects of the notebook.

---

**The central idea: fifty researchers, not fifty calculators**

The first thing to understand is that the fifty agents are not supposed to calculate option prices independently.

That would be redundant and not particularly interesting.

We already have a numerical pricing method in the notebook that can calculate reference American-option prices. The agents do not need to replace it.

Instead, the agents are being used for something closer to scientific reasoning.

Imagine that you had a difficult economics problem and invited specialists from different fields into the room.

One researcher might say:

“I think the most important issue is how close the option is to the point where exercising becomes attractive.”

Another might say:

“I think time remaining is critical.”

Another might say:

“You are ignoring dividends.”

Another might say:

“Whatever formula you construct must respect basic financial limits.”

Another might say:

“You are making the formula unnecessarily complicated. Strip it down.”

Another might say:

“This model looks fine under normal conditions, but what happens near expiration?”

Those are different intellectual contributions.

Cell 4 tries to reproduce that kind of diversity deliberately.

It does not create diversity by asking the same model the same question fifty times and hoping randomness will produce fifty different answers.

Instead, it gives the agents different jobs.

That is why the code says, in effect:

ten specialist families multiplied by five research lenses equals fifty agents.

The diversity comes from design.

---

**The Ten Specialist Families**

The first major source of diversity is the ten specialist families.

Each family represents a different intellectual tradition or way of looking at the American-option problem.

You do not need to be an options specialist to understand them.

Think of them as ten departments in the same research institute.

---

**1. Free-boundary specialists**

The first family thinks about what the code calls the “free boundary.”

That expression sounds intimidating, but the underlying idea is straightforward.

With an American option, there are situations in which it is better to keep the option alive and other situations in which it may be better to exercise it immediately.

Somewhere between those regions there is a dividing line.

On one side, waiting is preferable.

On the other side, exercising is preferable.

The location of that dividing line depends on things such as the stock price, the strike price, the remaining time, volatility, interest rates, and dividends.

The free-boundary specialists therefore ask:

“What variables help us understand how close the option is to the exercise region?”

They are interested in the geometry of the exercise decision.

Their instinct is that a good approximation for the American-option premium should somehow recognize whether the option is far away from early exercise, close to early exercise, or already deeply within a region where exercise may be attractive.

In simple terms, they focus on the border between “wait” and “exercise.”

---

**2. Asymptotics specialists**

The second family studies what happens in extreme or limiting situations.

“Asymptotics” is mathematical jargon for a very useful question:

“What happens when something becomes very small, very large, very short, or very long?”

For example, what happens when the option is only a few days away from expiration?

What happens when there is a great deal of time remaining?

What happens when the extra value of early exercise is almost zero?

What happens when the option becomes very deeply in or out of the money?

These agents are valuable because many formulas can look reasonable in ordinary conditions while behaving badly at the edges.

The asymptotics specialists act like researchers who test a theory by pushing it toward unusual but informative situations.

Their contribution is often not “here is the answer.”

Their contribution is:

“Whatever answer you build should behave sensibly when we approach these limits.”

---

**3. Arbitrage-bounds specialists**

The third family focuses on financial consistency.

The word “arbitrage” can sound highly technical, but here the central intuition is simple.

An option price cannot be just any number.

Basic financial logic imposes boundaries on what prices are economically possible.

For example, an American option should not normally be worth less than an otherwise equivalent European option, because the American contract gives the owner at least as much flexibility.

There are also upper limits on what a call or put can reasonably be worth.

The arbitrage-bounds agents therefore behave almost like accountants or constitutional lawyers.

They ask:

“Does this proposed structure obey the basic rules of the financial system?”

They are less interested in elegance and more interested in preventing nonsense.

Their purpose is to make sure that a compact formula does not accidentally imply economically impossible prices.

---

**4. Dividends-and-carry specialists**

This family concentrates on interest rates, dividends, and the cost or benefit of holding different financial positions over time.

The word “carry” refers broadly to the financial consequences of holding something through time.

For American options, dividends can be especially important.

A shareholder may receive dividends. An option holder generally does not receive those dividends unless the option is exercised and converted into the stock.

That means the presence of dividends can affect whether early exercise becomes attractive.

Interest rates can also influence the decision because money received or paid today has different value from money received or paid later.

These agents therefore ask questions such as:

“How strongly should dividends affect the early-exercise premium?”

“How does the relationship between interest rates and dividends matter?”

“Are there combinations of rates, dividends, and time that should appear together?”

Their role is to make sure the formula respects the economics of financing and distributions.

---

**5. Moneyness-geometry specialists**

“Moneyness” is another piece of derivatives jargon that sounds more complicated than it is.

It simply describes the relationship between the current stock price and the option’s strike price.

The strike price is the predetermined price at which the option gives you the right to buy or sell.

Depending on the relationship between the stock price and the strike price, the option may be comfortably valuable, only marginally valuable, or currently unattractive to exercise.

The moneyness specialists focus on that relative position.

They ask:

“How far is this option from the economically important region around its strike?”

They are not primarily interested in whether the stock is worth 50 dollars, 100 dollars, or 500 dollars.

They care about the relationship between stock price and strike price.

That is why the notebook constructs normalized variables rather than relying only on raw prices.

This family is trying to describe the shape of the problem.

---

**6. Maturity-regime specialists**

These agents concentrate on time.

An American option with two days remaining is not the same economic object as an otherwise similar option with two years remaining.

The importance of early exercise can change substantially depending on how much time remains.

Near expiration, decisions can become much sharper because there is little time for future uncertainty to resolve.

Far from expiration, waiting retains more value because there are many possible future paths for the stock.

The maturity specialists ask:

“Does the relationship between time and early exercise change across different time horizons?”

They are particularly interested in what the code calls “regimes.”

A regime is simply a region of circumstances in which behavior looks meaningfully different.

The idea is that short-dated options may behave differently from medium-dated or long-dated options.

These agents try to make sure that the final model is sensitive to those differences.

---

**7. Volatility-structure specialists**

Volatility is a measure of how much the underlying stock price tends to move around.

High volatility means a wider range of future stock prices is plausible.

Low volatility means future prices are expected to remain more concentrated.

For options, volatility is extremely important because options derive much of their value from uncertainty.

The volatility specialists ask:

“How does uncertainty interact with the early-exercise decision?”

Their role is especially interesting because volatility may not matter by itself.

It may matter differently depending on time remaining or on how far the option is from its strike price.

These agents therefore pay particular attention to combinations of variables.

They might suspect, for example, that volatility together with time is more informative than either variable considered separately.

Their job is to understand the structure of uncertainty.

---

**8. Symbolic-regression specialists**

This family is closest to what we might call formula designers.

Symbolic regression is the search for a mathematical expression that is both useful and relatively simple.

The important word here is “interpretable.”

We could potentially use a very large machine-learning model to approximate American-option prices.

But that would defeat the purpose of this notebook.

We are trying to discover a compact approximation that a human researcher can inspect.

The symbolic-regression agents therefore ask:

“What small combination of available ingredients might capture the important behavior?”

They are looking for structure, not black-box prediction.

Their instinct is to combine a few meaningful variables into something that could eventually resemble a traditional analytical approximation.

---

**9. Numerical-stability specialists**

These agents think like engineers.

A formula can look good theoretically and still be unpleasant to use.

For example, two variables may contain almost the same information. That can make fitted coefficients unstable.

A formula may work well for ordinary cases but behave erratically when inputs change slightly.

The numerical-stability agents ask:

“Can this proposed structure be estimated reliably?”

“Are these variables redundant?”

“Will the formula behave reasonably outside the exact observations used to construct it?”

They are concerned with robustness.

Their purpose is not to propose the most sophisticated formula.

They want one that survives contact with computation.

---

**10. Adversarial critics**

The final family is deliberately skeptical.

These agents are told, in effect:

“Assume the other researchers may be missing something.”

Their task is to challenge assumptions.

They look for failure modes.

They may ask:

“What happens if the option is extremely close to expiration?”

“What if dividends are unusually important?”

“What if a supposedly useful variable works only around normal market conditions?”

“What if two variables appear useful because they contain nearly the same information?”

These agents are a form of institutionalized criticism.

Instead of hoping somebody notices weaknesses later, the system explicitly creates agents whose job is to look for them.

That is a powerful general principle for agentic systems: criticism should be designed into the architecture.

---

**Why each family contains five agents**

The ten specialist families give us ten different theoretical perspectives.

But Cell 4 goes further.

Each family is instantiated five times using five different “variant lenses.”

This means that even within a specialty, the agents are instructed to think differently.

The first variant asks for the smallest defensible set of variables.

This agent behaves like a minimalist. It asks what can be removed.

The second variant concentrates on interactions.

An interaction means that the importance of one factor may depend on another factor.

For example, time may matter differently when volatility is high than when volatility is low.

The third variant focuses on limiting cases.

It asks what the proposed structure should do in unusual but revealing situations.

The fourth focuses on robustness.

It wants something that works across the full range of market conditions considered by the experiment.

The fifth acts as a skeptical coauthor.

It proposes a structure but is also required to identify how that structure might fail.

So even if five agents belong to the volatility family, they are not being asked exactly the same question.

One is a minimalist volatility researcher.

One is an interaction-oriented volatility researcher.

One is a limiting-case volatility researcher.

One is a robustness-oriented volatility researcher.

One is a skeptical volatility researcher.

The same design is repeated across all ten families.

That produces fifty distinct research mandates.

---

**What exactly do we give each agent?**

Every agent receives a carefully constructed research brief.

This is crucial.

We are not simply saying:

“Please solve American options.”

Each agent receives several pieces of information.

First, it receives its identity.

It is told which specialist family it belongs to and which of the five variant lenses it must adopt.

Second, it receives the research problem.

It is told that we are trying to understand the extra value associated with the early-exercise feature of American options.

Third, it receives the domain of the experiment.

This tells the agent what kinds of options the notebook is studying.

For example, it knows the ranges of maturity, volatility, interest rates, dividend yields, and relative stock-versus-strike positions.

This matters because a useful model for ordinary equity options does not necessarily need to solve every imaginable option-pricing situation in the universe.

Fourth, the agent receives a permitted vocabulary of candidate features.

This is one of the most important governance mechanisms in the notebook.

The agents are not free to invent arbitrary variables or executable formulas.

They must choose from a predefined list created by the notebook.

This makes their proposals comparable.

It also makes them auditable.

If fifty agents were allowed to invent fifty completely different mathematical languages, combining their recommendations would become extremely difficult.

Instead, we give them a common vocabulary and ask them to vote, implicitly, on which concepts matter most.

Fifth, every agent is told what it is not allowed to do.

It must not estimate coefficients.

It must not claim to know the numerical accuracy of its idea.

It must not invent performance statistics.

It must not pretend it has discovered an exact analytical solution.

That separation is deliberate.

The agents propose structure.

The numerical methods later determine truth.

---

**What does “training” mean here?**

This is probably the most important piece of terminology to clarify.

In this notebook, “training” does not mean that we are training Claude.

We are not changing Claude's internal neural network.

We are not teaching the fifty agents derivatives from scratch.

Claude already arrives as a pretrained language model.

The word “training” here refers to the numerical model that comes later.

Earlier in the notebook, we generated thousands of artificial option contracts.

For each contract, we calculated a reference American-option price using a numerical tree.

We then divided those contracts into two groups.

One group is the training set.

The other is the holdout or test set.

The training set is the portion of data that the notebook is allowed to use when estimating the coefficients of the eventual approximation.

In ordinary language, think of it as the workshop.

The model is allowed to look at those examples while figuring out how strongly different selected variables should matter.

The test set is different.

It is kept aside.

Think of it as the examination.

The model does not get to use those observations when choosing its coefficients.

Only after the formula has been built do we expose it to those unseen cases.

That tells us whether the approximation has learned something general or merely adapted itself to the examples it already saw.

There is therefore an important distinction between two kinds of “learning” happening in the overall experiment.

The agents contribute conceptual learning.

They propose which variables might matter.

The numerical fitting procedure performs statistical learning.

It estimates how much those variables actually matter based on the training data.

And the holdout set provides the independent exam.

---

**The deeper purpose of Cell 4**

Cell 4 is really an experiment in organizing intelligence.

We could have asked one powerful language model:

“What variables should I use to approximate American-option early-exercise value?”

Instead, we deliberately construct a miniature scientific institution.

There are specialists.

There are different research philosophies.

There is a shared vocabulary.

There are limits on what researchers are allowed to claim.

There is a separation between hypothesis generation and empirical verification.

There are critics.

And later, there will be collective voting and numerical testing.

That is why the fifty-agent architecture is interesting beyond option pricing.

The agents are not valuable because fifty is automatically better than one.

They are valuable because they are organized.

Cell 4 gives us ten forms of expertise, five styles of inquiry within each expertise, one common research problem, one common feature vocabulary, one common experimental domain, and one disciplined output format.

The result is not fifty answers.

It is fifty structured hypotheses.

The notebook can then ask a much more interesting question:

When many differently instructed artificial researchers examine the same problem, which ideas repeatedly survive?

And once those ideas are identified, do they actually improve a compact numerical approximation when tested against independent data?

That is the bridge between the agentic part of the notebook and the quantitative-finance part.

Cell 4 creates the intellectual diversity.

Cell 5 lets those agents speak.

Cell 6 measures where they agree.

The later cells allow the data to decide whether their collective intuition was useful.

So the simplest interpretation of Cell 4 is this:

We are assembling a research team.

We give every member the same problem and the same factual boundaries.

We give different members different specialties and different intellectual attitudes.

We prevent them from pretending they have already solved the problem.

We ask each one for a short, disciplined hypothesis.

Then we send those hypotheses downstream to a completely separate numerical system that will decide what deserves to survive.

That is what Cell 4 is doing.

## CELL 5 — Parallel swarm execution with structured JSON contracts

Now the fifty agents are allowed to act. The notebook executes their prompts concurrently, subject to a conservative worker limit. Parallelism matters because the agents are conceptually independent hypothesis generators; serial execution would add latency without adding information. Each request asks Claude Sonnet 5 for a compact JSON object rather than free-form prose.

The execution wrapper is defensive. It extracts JSON even if the model surrounds it with incidental text, records failures rather than hiding them, and preserves the family, variant, and raw response for provenance. API errors do not crash the entire swarm. This is important in real multi-agent systems: partial failure should degrade the sample size, not destroy the experiment.

The language model is used only for the task at which it is strongest here—reasoning over candidate mathematical structure. It does not receive authority to execute arbitrary code, alter the benchmark, or write coefficients into the final formula. The temperature is kept low because diversity comes from mandates. This avoids confusing randomness with expertise.

At the end of the unit, the notebook reports how many agents returned parseable contracts and how many failed. Token usage is retained when available. A serious swarm experiment should expose its computational footprint and failure rate. If too few agents succeed, the notebook should be rerun or the API configuration repaired before moving to consensus. The downstream cells can tolerate missing agents, but they should not disguise a severely incomplete swarm as a fifty-agent result.

In [13]:
# CELL 5 — Parallel swarm execution with robust structured outputs
#
# Claude Sonnet 5 / current Anthropic Messages API
#
# IMPORTANT:
# - no temperature
# - no top_p
# - no top_k
# - structured JSON output
# - explicit stop_reason checking
# - automatic retry on truncated structured output
# - one-agent pre-flight before launching the swarm


# ============================================================
# TOKEN BUDGETS
# ============================================================

INITIAL_MAX_TOKENS = 4096
RETRY_MAX_TOKENS = 8192


# ============================================================
# STRUCTURED OUTPUT CONTRACT
# ============================================================

AGENT_OUTPUT_SCHEMA = {

    "type": "object",

    "properties": {

        "family": {
            "type": "string",
            "description":
                "The specialist family assigned to the agent."
        },

        "variant": {
            "type": "integer",
            "description":
                "The assigned variant number."
        },

        "features": {

            "type": "array",

            "description":
                "A compact set of candidate features from the permitted vocabulary.",

            "items": {
                "type": "string"
            }
        },

        "must_have_interactions": {

            "type": "array",

            "description":
                "Especially important interaction terms from the permitted vocabulary.",

            "items": {
                "type": "string"
            }
        },

        "rationale": {

            "type": "string",

            "description":
                "Concise mathematical and economic rationale, ideally no more than 100 words."
        },

        "expected_failure_modes": {

            "type": "array",

            "description":
                "Up to three concise situations in which the proposed structure may fail.",

            "items": {
                "type": "string"
            }
        },

        "confidence": {

            "type": "number",

            "description":
                "Subjective structural confidence between 0 and 1."
        },
    },

    "required": [

        "family",
        "variant",
        "features",
        "must_have_interactions",
        "rationale",
        "expected_failure_modes",
        "confidence",
    ],

    "additionalProperties": False,
}


# ============================================================
# LOW-LEVEL CLAUDE CALL
# ============================================================

def call_claude_agent(
    spec,
    max_tokens
):

    msg = client.messages.create(

        model=MODEL,

        max_tokens=max_tokens,

        system=(
            "You are a rigorous quantitative-finance research agent. "
            "You perform structural mathematical hypothesis generation only. "
            "Follow the assigned mandate precisely. "
            "Be concise. "
            "Do not estimate option prices or coefficients. "
            "Your response must conform exactly to the supplied "
            "structured-output schema."
        ),

        messages=[
            {
                "role": "user",
                "content": spec["prompt"],
            }
        ],

        output_config={

            "format": {

                "type": "json_schema",

                "schema": AGENT_OUTPUT_SCHEMA,
            }
        },
    )

    return msg


# ============================================================
# EXTRACT TEXT SAFELY
# ============================================================

def extract_text_blocks(msg):

    blocks = [

        block.text

        for block in msg.content

        if getattr(
            block,
            "type",
            None
        ) == "text"

        and hasattr(
            block,
            "text"
        )
    ]

    return "".join(blocks).strip()


# ============================================================
# ONE AGENT
# ============================================================

def run_agent(spec):

    t0 = time.time()

    attempts = []


    try:

        # ----------------------------------------------------
        # ATTEMPT 1
        # ----------------------------------------------------

        msg = call_claude_agent(
            spec,
            INITIAL_MAX_TOKENS
        )

        raw = extract_text_blocks(msg)

        attempts.append(
            {
                "max_tokens":
                    INITIAL_MAX_TOKENS,

                "stop_reason":
                    msg.stop_reason,

                "output_tokens":
                    getattr(
                        msg.usage,
                        "output_tokens",
                        None
                    ),
            }
        )


        # ----------------------------------------------------
        # IMPORTANT:
        # NEVER PARSE STRUCTURED JSON BEFORE CHECKING WHETHER
        # GENERATION WAS TRUNCATED.
        # ----------------------------------------------------

        if msg.stop_reason == "max_tokens":

            print(
                f"Retrying "
                f"{spec['family']}-{spec['variant']} "
                f"because structured output hit "
                f"{INITIAL_MAX_TOKENS} tokens."
            )


            # ------------------------------------------------
            # ATTEMPT 2 — GENERATE AGAIN WITH MORE ROOM
            # ------------------------------------------------

            msg = call_claude_agent(
                spec,
                RETRY_MAX_TOKENS
            )

            raw = extract_text_blocks(msg)

            attempts.append(
                {
                    "max_tokens":
                        RETRY_MAX_TOKENS,

                    "stop_reason":
                        msg.stop_reason,

                    "output_tokens":
                        getattr(
                            msg.usage,
                            "output_tokens",
                            None
                        ),
                }
            )


        # ----------------------------------------------------
        # CHECK FINAL STOP CONDITION
        # ----------------------------------------------------

        if msg.stop_reason == "max_tokens":

            raise RuntimeError(
                "Structured response was still truncated "
                f"after retry with {RETRY_MAX_TOKENS} tokens."
            )


        if msg.stop_reason == "refusal":

            raise RuntimeError(
                "Claude refused the request."
            )


        if msg.stop_reason not in (
            "end_turn",
            None,
        ):

            raise RuntimeError(
                "Unexpected Anthropic stop_reason: "
                f"{msg.stop_reason}"
            )


        # ----------------------------------------------------
        # EMPTY RESPONSE CHECK
        # ----------------------------------------------------

        if not raw:

            raise ValueError(
                "Anthropic returned an empty text response."
            )


        # ----------------------------------------------------
        # PARSE STRUCTURED JSON
        # ----------------------------------------------------

        try:

            obj = json.loads(raw)

        except json.JSONDecodeError as e:

            # Keep a useful diagnostic without dumping
            # an enormous model response.

            tail = raw[-500:]

            raise ValueError(

                "Structured output completed but JSON parsing failed.\n"
                f"stop_reason={msg.stop_reason}\n"
                f"raw_length={len(raw)} characters\n"
                f"last_500_chars={tail!r}\n"
                f"original_error={repr(e)}"
            )


        # ----------------------------------------------------
        # AUTHORITATIVE AGENT IDENTITY
        # ----------------------------------------------------
        #
        # We trust the orchestration layer—not the model—to
        # determine family and variant identity.
        # ----------------------------------------------------

        obj["family"] = spec["family"]
        obj["variant"] = spec["variant"]


        # ----------------------------------------------------
        # RETURN SUCCESS
        # ----------------------------------------------------

        return {

            "ok": True,

            "family":
                spec["family"],

            "variant":
                spec["variant"],

            "obj":
                obj,

            "raw":
                raw,

            "seconds":
                time.time() - t0,

            "input_tokens":
                getattr(
                    msg.usage,
                    "input_tokens",
                    None
                ),

            "output_tokens":
                getattr(
                    msg.usage,
                    "output_tokens",
                    None
                ),

            "stop_reason":
                msg.stop_reason,

            "attempt_count":
                len(attempts),

            "attempts":
                attempts,

            "request_id":
                getattr(
                    msg,
                    "_request_id",
                    None
                ),
        }


    except Exception as e:

        return {

            "ok": False,

            "family":
                spec["family"],

            "variant":
                spec["variant"],

            "error":
                repr(e),

            "seconds":
                time.time() - t0,

            "attempt_count":
                len(attempts),

            "attempts":
                attempts,
        }


# ============================================================
# ONE-AGENT PRE-FLIGHT
# ============================================================

print("=" * 72)
print("ANTHROPIC API PRE-FLIGHT")
print("=" * 72)


preflight = run_agent(
    prompts[0]
)


if not preflight["ok"]:

    print("PRE-FLIGHT FAILED")
    print()

    print(
        preflight["error"]
    )

    print()

    print(
        "Attempts:"
    )

    print(
        json.dumps(
            preflight.get(
                "attempts",
                []
            ),
            indent=2
        )
    )

    raise RuntimeError(
        "Anthropic API pre-flight failed. "
        "The 50-agent swarm was NOT launched."
    )


print("Pre-flight successful.")

print()

print(
    "Agent:",
    preflight["family"],
    "variant",
    preflight["variant"]
)

print(
    "Stop reason:",
    preflight["stop_reason"]
)

print(
    "Attempts:",
    preflight["attempt_count"]
)

print(
    "Output tokens:",
    preflight["output_tokens"]
)

print(
    "Execution time:",
    f"{preflight['seconds']:.2f} sec"
)

print()

print(
    "Returned fields:",
    list(
        preflight["obj"].keys()
    )
)


# ============================================================
# CONTRACT PREVIEW
# ============================================================

print()
print("=" * 72)
print("PRE-FLIGHT CONTRACT")
print("=" * 72)

print(
    json.dumps(
        preflight["obj"],
        indent=2,
        ensure_ascii=False,
    )
)


# ============================================================
# EXECUTE REMAINING 49 AGENTS
# ============================================================

print()
print("=" * 72)
print("LAUNCHING 50-AGENT RESEARCH SWARM")
print("=" * 72)


# Pre-flight agent counts as agent #1.
results = [
    preflight
]


with ThreadPoolExecutor(
    max_workers=10
) as executor:

    future_map = {

        executor.submit(
            run_agent,
            spec
        ): spec

        for spec in prompts[1:]
    }


    completed = 1


    for future in as_completed(
        future_map
    ):

        result = future.result()

        results.append(
            result
        )

        completed += 1


        if (
            completed % 5 == 0
            or completed == N_AGENTS
        ):

            success_count = sum(
                r["ok"]
                for r in results
            )

            failure_count = (
                len(results)
                -
                success_count
            )

            print(
                f"Completed {completed:2d}/{N_AGENTS} "
                f"| successful={success_count:2d} "
                f"| failures={failure_count:2d}"
            )


# ============================================================
# PARTITION RESULTS
# ============================================================

ok = [
    r
    for r in results
    if r["ok"]
]

bad = [
    r
    for r in results
    if not r["ok"]
]


# ============================================================
# FINAL EXECUTION SUMMARY
# ============================================================

print()
print("=" * 72)
print("SWARM EXECUTION SUMMARY")
print("=" * 72)

print(
    f"Successful contracts: "
    f"{len(ok)}/{len(results)}"
)

print(
    f"Failures:             "
    f"{len(bad)}"
)


# ============================================================
# USAGE STATISTICS
# ============================================================

if ok:

    total_input_tokens = sum(

        r.get(
            "input_tokens"
        ) or 0

        for r in ok
    )


    total_output_tokens = sum(

        r.get(
            "output_tokens"
        ) or 0

        for r in ok
    )


    mean_seconds = np.mean(
        [
            r["seconds"]
            for r in ok
        ]
    )


    median_seconds = np.median(
        [
            r["seconds"]
            for r in ok
        ]
    )


    retried_agents = sum(

        r.get(
            "attempt_count",
            1
        ) > 1

        for r in ok
    )


    print(
        f"Total input tokens:   "
        f"{total_input_tokens:,}"
    )

    print(
        f"Total output tokens:  "
        f"{total_output_tokens:,}"
    )

    print(
        f"Mean agent time:      "
        f"{mean_seconds:.2f} sec"
    )

    print(
        f"Median agent time:    "
        f"{median_seconds:.2f} sec"
    )

    print(
        f"Retried agents:       "
        f"{retried_agents}"
    )


# ============================================================
# FAILURE REPORT
# ============================================================

if bad:

    print()
    print("=" * 72)
    print("FAILED AGENTS")
    print("=" * 72)


    failure_rows = []


    for r in bad:

        failure_rows.append(
            {
                "family":
                    r["family"],

                "variant":
                    r["variant"],

                "error":
                    r["error"],

                "attempt_count":
                    r.get(
                        "attempt_count"
                    ),

                "seconds":
                    r["seconds"],
            }
        )


    failure_df = pd.DataFrame(
        failure_rows
    )

    display(
        failure_df
    )


else:

    print()

    print(
        "All 50 agents returned valid "
        "structured contracts."
    )

ANTHROPIC API PRE-FLIGHT
Pre-flight successful.

Agent: free_boundary variant 1
Stop reason: end_turn
Attempts: 1
Output tokens: 449
Execution time: 7.48 sec

Returned fields: ['family', 'variant', 'features', 'must_have_interactions', 'rationale', 'expected_failure_modes', 'confidence']

PRE-FLIGHT CONTRACT
{
  "family": "free_boundary",
  "variant": 1,
  "features": [
    "carryT",
    "put_itm",
    "call_itm",
    "absx_sig",
    "varT",
    "sqrtT_r",
    "sqrtT_q"
  ],
  "must_have_interactions": [
    "absx_sig",
    "carryT",
    "put_itm"
  ],
  "rationale": "Optimal-exercise premium is driven by the sign/magnitude of carry (r-q) and moneyness, since the free boundary exists only when carry favors early exercise (puts: r>0 dominates; calls: q>0 dominates). Using itm-hinge terms separates put/call exercise regions cheaply, while absx_sig captures the diffusive erosion of the exercise value near the boundary. Smallest defensible basis: linear carry, hinge moneyness, and one vola

## CELL 6 — Contract validation, provenance, and family-level consensus

Raw model output is not yet evidence. This unit validates each response against the common contract. Unknown features are removed, duplicate features are collapsed, complexity is capped, confidence is clipped to a valid interval, and missing textual fields receive explicit placeholders. The resulting tidy table is the canonical record of what the swarm actually proposed.

Consensus is computed in two ways. Raw support counts how many individual agents selected a feature. Family support counts how many of the ten specialist families selected it at least once. The second measure is especially important because five variants inside one family are correlated by design. A feature supported by six intellectually distinct families is more persuasive than one supported by five variants of a single specialty.

The notebook converts these votes into a ranked feature table and calculates a simple disagreement measure. It also creates a family-by-feature support matrix. These outputs make the swarm inspectable: one can see whether rate terms are being driven by carry specialists, whether asymptotic agents converge on square-root maturity, or whether critics identify interactions ignored by the rest.

The cell then selects a compact candidate basis using explicit thresholds, with a deterministic fallback to the most broadly supported features if the threshold would produce too small a model. This is the first point at which collective reasoning changes the numerical experiment. Importantly, it changes only the set of candidate explanatory terms. Coefficients and accuracy remain entirely determined by data.

In [14]:
# CELL 6 — Contract validation, provenance, and family-level consensus
def validate_contract(r):
    o = r["obj"]
    feats = [f for f in o.get("features", []) if f in ALLOWED_FEATURES]
    ints = [f for f in o.get("must_have_interactions", []) if f in ALLOWED_FEATURES]
    feats = list(dict.fromkeys(feats + ints))[:MAX_FEATURES_PER_AGENT]
    conf = float(o.get("confidence", 0.5))
    conf = min(max(conf, 0.0), 1.0)
    return {
        "family": r["family"],
        "variant": r["variant"],
        "features": feats,
        "rationale": str(o.get("rationale","")),
        "failure_modes": o.get("expected_failure_modes", []),
        "confidence": conf,
        "seconds": r["seconds"],
        "input_tokens": r.get("input_tokens"),
        "output_tokens": r.get("output_tokens"),
    }

contracts = pd.DataFrame([validate_contract(r) for r in ok])
if len(contracts) < 10:
    raise RuntimeError("Too few valid agents. Repair API execution and rerun CELL 5.")

rows = []
for _, row in contracts.iterrows():
    for f in row["features"]:
        rows.append({"family":row["family"],"variant":row["variant"],"feature":f,"confidence":row["confidence"]})
votes = pd.DataFrame(rows)

support = votes.groupby("feature").agg(
    agent_support=("variant","size"),
    family_support=("family","nunique"),
    mean_confidence=("confidence","mean")
).sort_values(["family_support","agent_support"], ascending=False)

family_matrix = (votes.assign(v=1)
                 .pivot_table(index="family", columns="feature", values="v", aggfunc="max", fill_value=0))

eligible = support[(support["family_support"] >= 3) | (support["agent_support"] >= max(5, len(contracts)//5))]
selected_features = eligible.head(MAX_FINAL_FEATURES).index.tolist()
if len(selected_features) < 6:
    selected_features = support.head(min(10, len(support))).index.tolist()

print("Selected swarm basis:", selected_features)
display(support.head(20))
display(family_matrix[selected_features])

plt.figure(figsize=(10,5))
support.head(15)["family_support"].sort_values().plot(kind="barh")
plt.xlabel("Number of specialist families supporting feature")
plt.title("Swarm consensus by independent specialist family")
plt.tight_layout()
plt.show()

Selected swarm basis: ['call_itm', 'put_itm', 'varT', 'sig_sqrtT', 'carryT', 'qT', 'rT', 'x_rT', 'disc_q', 'disc_r', 'x_qT', 'sqrtT_q', 'sqrtT_r', 'call_itm2']


,agent_support,family_support,mean_confidence
feature,,,
call_itm,50,10,0.611400
put_itm,50,10,0.611400
varT,50,10,0.611400
sig_sqrtT,46,10,0.610652
carryT,35,10,0.616857
qT,29,10,0.605862
rT,29,10,0.605862
x_rT,33,9,0.606970
disc_q,25,9,0.614000


feature,call_itm,put_itm,varT,sig_sqrtT,carryT,qT,rT,x_rT,disc_q,disc_r,x_qT,sqrtT_q,sqrtT_r,call_itm2
family,,,,,,,,,,,,,,
adversarial_critic,1,1,1,1,1,1,1,1,1,1,1,0,0,0
arbitrage_bounds,1,1,1,1,1,1,1,1,1,1,1,1,1,0
asymptotics,1,1,1,1,1,1,1,1,1,1,1,1,1,1
dividends_carry,1,1,1,1,1,1,1,1,1,1,1,0,0,1
free_boundary,1,1,1,1,1,1,1,1,1,1,1,1,1,1
maturity_regimes,1,1,1,1,1,1,1,1,0,0,1,1,1,1
moneyness_geometry,1,1,1,1,1,1,1,1,1,1,0,1,1,1
numerical_stability,1,1,1,1,1,1,1,1,1,1,1,0,0,1
symbolic_regression,1,1,1,1,1,1,1,1,1,1,1,1,1,0


<Figure size 1000x500 with 1 Axes>

## CELL 7 — Fit the swarm-selected early-exercise-premium equation

The seventh unit translates collective structural hypotheses into an actual approximate equation. The dependent variable is the normalized early-exercise premium, $(V_A-V_E)/K$. Separate models are estimated for puts and dividend-paying calls because the economics of early exercise differs materially between them. Non-dividend-paying calls are handled by the exact equality $C_A=C_E$.

For each option type, the design matrix contains an intercept and the features selected by the swarm. Coefficients are estimated with a small ridge penalty. Ridge regularization is used not because the notebook seeks a black-box predictive model, but because nonlinear feature libraries can contain correlated terms. A small penalty stabilizes coefficients and reduces the risk that an apparently compact formula is numerically fragile.

The raw fitted premium is passed through a positive-part operator. The reconstructed American price is then projected onto elementary no-arbitrage bounds: it cannot fall below either European value or intrinsic value, and it cannot exceed spot for calls or strike for puts. These projections are transparent and deterministic.

Most importantly, the cell prints the fitted equation in human-readable mathematical form. The output is not a hidden estimator object. It is a finite expression of the form European Black–Scholes plus strike times a positive part of a weighted feature sum. That expression is the central artifact of the notebook. The remaining units are devoted to trying to falsify it.

In [17]:
# CELL 7 — Fit the swarm-selected early-exercise-premium equation
models = {}

def fit_type(option_type, features):
    d = train[train["type"] == option_type].copy()
    if option_type == "call":
        d = d[d["q"] > 1e-8]  # q=0 calls handled by exact equality
    Z = feature_frame(d)[features]
    X = np.column_stack([np.ones(len(Z)), Z.values])
    y = d["premium_norm"].values
    model = Ridge(alpha=RIDGE_ALPHA, fit_intercept=False)
    model.fit(X, y)
    return model

for typ in ["put","call"]:
    models[typ] = fit_type(typ, selected_features)

def predict_american(d):
    out = np.zeros(len(d))
    Zall = feature_frame(d)[selected_features]
    for i, (_, row) in enumerate(d.iterrows()):
        typ = row["type"]
        euro = row["euro"] if "euro" in row else bs_price(row.S,row.K,row.T,row.r,row.q,row.sigma,typ)
        intrinsic = max(row.S-row.K,0) if typ=="call" else max(row.K-row.S,0)
        if typ == "call" and row.q <= 1e-10:
            pred = euro
        else:
            z = Zall.iloc[i].values
            raw_norm = models[typ].predict(np.r_[1.0,z].reshape(1,-1))[0]
            premium = row.K * max(0.0, raw_norm)
            pred = euro + premium
        lower = max(euro, intrinsic)
        upper = row.S if typ=="call" else row.K
        out[i] = min(max(pred, lower), upper)
    return out

def equation_string(typ):
    coef = models[typ].coef_
    terms = [f"{coef[0]:+.8g}"]
    for c, f in zip(coef[1:], selected_features):
        terms.append(f"{c:+.8g}*{f}")
    return " ".join(terms)

print("PUT:")
print("P_A ≈ max(bounds, P_BS + K * max(0,", equation_string("put"), "))")
print("\nDIVIDEND-PAYING CALL:")
print("C_A ≈ max(bounds, C_BS + K * max(0,", equation_string("call"), "))")
print("\nNON-DIVIDEND CALL: C_A = C_BS")

PUT:
P_A ≈ max(bounds, P_BS + K * max(0, +0.62910621 +0.00091456021*call_itm +0.0092116106*put_itm -0.0015528651*varT -0.0010236153*sig_sqrtT +0.12367281*carryT -0.27701918*qT -0.15334637*rT +0.51810165*x_rT -0.43895037*disc_q -0.18999585*disc_r -0.281887*x_qT -0.097047118*sqrtT_q -0.013540764*sqrtT_r +0.0095093008*call_itm2 ))

DIVIDEND-PAYING CALL:
C_A ≈ max(bounds, C_BS + K * max(0, +0.90221663 +0.0077113913*call_itm +0.006049391*put_itm +0.0064287521*varT -0.0075884689*sig_sqrtT -0.043106706*carryT -0.32524121*qT -0.36834791*rT +0.24228089*x_rT -0.43378504*disc_q -0.46831902*disc_r -0.63351367*x_qT +0.026812647*sqrtT_q -0.10788213*sqrtT_r +0.016513749*call_itm2 ))

NON-DIVIDEND CALL: C_A = C_BS


## CELL 8 — Holdout validation, regime diagnostics, and arbitrage checks

An approximation earns credibility on data that did not determine its coefficients. This unit therefore evaluates the equation on the holdout sample and reports multiple error statistics: mean absolute error, root mean squared error, median absolute error, the 95th percentile of absolute error, and error normalized by strike. No single metric is sufficient. RMSE exposes large misses, median error describes typical performance, and tail error shows whether a few difficult contracts dominate risk.

The notebook also slices errors by economically meaningful regimes. Short-dated contracts are separated from longer maturities; high-volatility contracts are isolated; deep in-the-money and deep out-of-the-money regions are compared with near-the-money observations; dividend-paying calls receive special attention. These diagnostics matter because the free-boundary problem is not uniformly difficult across the state space.

Arbitrage diagnostics are reported separately from statistical errors. The reconstructed price is checked against intrinsic value, European value, and simple upper bounds. Since the final prediction function explicitly projects onto those bounds, violations should be zero up to numerical tolerance. That is a design feature, not evidence of predictive excellence, and the notebook states the distinction.

Finally, the cell identifies the worst holdout observations and prints their parameters. A useful research notebook should make failure cases easy to inspect rather than burying them in an average score. These contracts become natural targets for later extensions of the feature library or specialized regime equations.

In [18]:
# CELL 8 — Holdout validation, regime diagnostics, and arbitrage checks
test = test.copy()
test["pred"] = predict_american(test)
test["abs_err"] = np.abs(test["pred"] - test["american"])
test["sq_err"] = (test["pred"] - test["american"])**2
test["abs_err_K"] = test["abs_err"] / test["K"]
test["x"] = np.log(test["K"]/test["S"])

def metrics(d):
    return pd.Series({
        "N": len(d),
        "MAE": d["abs_err"].mean(),
        "RMSE": np.sqrt(d["sq_err"].mean()),
        "MedianAE": d["abs_err"].median(),
        "P95_AE": d["abs_err"].quantile(.95),
        "MAE_over_K": d["abs_err_K"].mean(),
    })

summary_metrics = pd.DataFrame({
    "ALL": metrics(test),
    "PUT": metrics(test[test.type=="put"]),
    "CALL": metrics(test[test.type=="call"]),
}).T
display(summary_metrics)

regimes = {
    "short_T": test["T"] < 0.10,
    "high_vol": test["sigma"] > 0.45,
    "deep_ITM_or_OTM": np.abs(test["x"]) > 0.22,
    "near_ATM": np.abs(test["x"]) < 0.05,
    "dividend_calls": (test["type"]=="call") & (test["q"]>0.02),
}
regime_table = pd.DataFrame({name: metrics(test[mask]) for name,mask in regimes.items()}).T
display(regime_table)

intrinsic = np.where(test.type=="call", np.maximum(test.S-test.K,0), np.maximum(test.K-test.S,0))
upper = np.where(test.type=="call", test.S, test.K)
violations = pd.Series({
    "below_intrinsic": int(np.sum(test.pred < intrinsic-1e-8)),
    "below_european": int(np.sum(test.pred < test.euro-1e-8)),
    "above_simple_upper_bound": int(np.sum(test.pred > upper+1e-8)),
})
print("Arbitrage-bound violations:\n", violations)

print("\nWorst holdout cases:")
display(test.nlargest(12,"abs_err")[["type","S","K","T","r","q","sigma","american","pred","abs_err"]])

,N,MAE,RMSE,MedianAE,P95_AE,MAE_over_K
ALL,550.0,0.104729,0.191275,0.035479,0.401885,0.001017
PUT,267.0,0.114108,0.206522,0.030767,0.440876,0.001042
CALL,283.0,0.095881,0.175680,0.039434,0.350096,0.000994


,N,MAE,RMSE,MedianAE,P95_AE,MAE_over_K
short_T,183.0,0.059678,0.100360,0.014878,0.214853,0.000561
high_vol,167.0,0.086204,0.157616,0.030548,0.310035,0.000848
deep_ITM_or_OTM,204.0,0.134973,0.210715,0.089606,0.437650,0.001308
near_ATM,80.0,0.089626,0.199880,0.006101,0.439283,0.000897
dividend_calls,217.0,0.113822,0.196537,0.061240,0.390595,0.001184


Arbitrage-bound violations:
 below_intrinsic             0
below_european              0
above_simple_upper_bound    0
dtype: int64

Worst holdout cases:


,type,S,K,T,r,q,sigma,american,pred,abs_err
1526,put,100.0,112.849501,2.880384,0.053403,0.065159,0.135998,17.413875,18.620311,1.206437
1186,call,100.0,133.103952,2.542405,0.000651,0.079353,0.372128,8.012017,7.000178,1.011839
1806,call,100.0,101.917609,2.745811,0.004896,0.076024,0.120138,2.605693,3.589337,0.983644
899,call,100.0,139.442909,2.643396,0.078502,0.074700,0.528350,19.865286,18.966751,0.898535
1170,put,100.0,115.985239,2.541287,0.036683,0.048278,0.230017,24.721809,25.530568,0.808759
1803,put,100.0,94.054981,2.426537,0.049080,0.039238,0.106966,2.998826,3.799429,0.800603
677,put,100.0,71.444504,2.648972,0.075614,0.035720,0.485985,10.755699,9.982726,0.772973
1916,call,100.0,101.214342,2.982160,0.047224,0.057410,0.184375,9.905229,10.667785,0.762555
1503,put,100.0,125.141313,2.120627,0.052627,0.040841,0.430395,38.209194,38.886764,0.677570
296,put,100.0,129.973352,1.774895,0.062897,0.028534,0.416232,37.875842,38.540099,0.664257




CELL 8 performs crucial validation checks on the fitted American option approximation. It evaluates the model's performance on a holdout dataset (data not used for training) to assess its generalization ability, examines performance across different market regimes, and checks for arbitrage violations.

**1. Summary Metrics (`summary_metrics`)**
This table provides an overview of the model's accuracy across all option types, as well as separately for 'put' and 'call' options. The key metrics reported are:

*   **N**: The number of options in each category.
*   **MAE (Mean Absolute Error)**: The average absolute difference between the predicted American option price (`pred`) and the true American option price (`american`). A lower MAE indicates better average accuracy.
*   **RMSE (Root Mean Squared Error)**: Similar to MAE, but it penalizes larger errors more heavily. A lower RMSE is desirable.
*   **MedianAE (Median Absolute Error)**: The median of the absolute errors. This metric is less sensitive to outliers than MAE or RMSE and gives a good sense of typical performance.
*   **P95_AE (95th Percentile Absolute Error)**: The absolute error below which 95% of all errors fall. This helps to understand the tail performance of the model and identify the magnitude of the largest, but still typical, errors.
*   **MAE_over_K (Mean Absolute Error over Strike)**: The MAE normalized by the strike price (`K`). This is useful for comparing error magnitudes across options with different strike prices, as absolute errors tend to be larger for higher-priced options.

By comparing these metrics across 'ALL', 'PUT', and 'CALL' categories, we can see if the model performs significantly better or worse for a particular option type.

**2. Regime Diagnostics (`regime_table`)**
This table breaks down the performance metrics (MAE, RMSE, etc.) for specific market regimes. This is critical because options pricing can be more challenging in certain conditions. The regimes analyzed are:

*   **`short_T`**: Options with short maturities (e.g., `T < 0.10`). These can be more sensitive to small changes in parameters.
*   **`high_vol`**: Options with high implied volatility (e.g., `sigma > 0.45`). High volatility typically means larger price movements, making pricing more complex.
*   **`deep_ITM_or_OTM`**: Options that are deep in-the-money or deep out-of-the-money (e.g., `|x| > 0.22`, where `x` is related to moneyness). These regions can have different early exercise behaviors.
*   **`near_ATM`**: Options that are close to at-the-money (e.g., `|x| < 0.05`). These are often the most liquid and actively traded options.
*   **`dividend_calls`**: Call options that pay a significant dividend (e.g., `q > 0.02`). Dividends affect the early exercise decision for calls.

Examining this table helps identify if the model struggles systematically in particular market conditions, even if its overall average performance is good. For instance, if `MAE` is significantly higher for `short_T` options, it suggests a weakness in modeling short-dated contracts.

**3. Arbitrage-Bound Violations (`violations`)**
This section reports the number of times the predicted American option price violates fundamental no-arbitrage bounds. The model explicitly projects its predictions onto these bounds (`min(max(pred, lower), upper)`), so the counts should ideally be zero.

*   **`below_intrinsic`**: Number of predictions that are lower than the intrinsic value of the option. An option should never trade below its intrinsic value.
*   **`below_european`**: Number of predictions that are lower than the European option price. An American option should always be worth at least as much as its European counterpart.
*   **`above_simple_upper_bound`**: Number of predictions that exceed a simple upper bound (spot price for calls, strike price for puts). Options cannot be worth more than this.

Zero violations here indicates that the model's final projected predictions respect these basic financial principles, which is a design feature rather than a measure of predictive power alone.

**4. Worst Holdout Cases (`Worst holdout cases`)**
This table displays the 12 observations from the holdout set where the model exhibited the largest absolute errors. For each of these cases, it shows:

*   **`type`**: The option type (put or call).
*   **`S`, `K`, `T`, `r`, `q`, `sigma`**: The input parameters (spot price, strike, time to maturity, risk-free rate, dividend yield, volatility).
*   **`american`**: The true American option price (from the CRR tree).
*   **`pred`**: The model's predicted American option price.
*   **`abs_err`**: The absolute error for that specific observation.

Inspecting these worst cases is crucial for understanding the model's limitations. By looking at the parameters of these 'failed' predictions, researchers can gain insights into where the current feature set or model structure might be insufficient and identify areas for future improvement. For example, if many worst cases share extreme values for `T` or `sigma`, it points to those regions as problematic.

## CELL 9 — Swarm-size convergence: do 50 agents add information?

A fifty-agent architecture should justify its size. This unit asks whether the selected mathematical structure stabilizes as the swarm grows. It constructs nested subswarms of 5, 10, 20, 30, 40, and 50 validated agents, recomputes feature support at each size, fits the corresponding deterministic approximation on the same training set, and evaluates it on the same holdout set.

The exercise is not a horse race in which the largest swarm is assumed to win. More agents can add genuinely different structural ideas, but they can also add correlated noise. The relevant evidence is convergence: do the selected features settle into a stable core, does holdout error improve materially, and does family coverage broaden? If performance stops changing after twenty or thirty agents, that is an important result. It suggests that the marginal informational value of additional agents is low under this mandate design.

Because subswarm order could matter, the notebook uses a deterministic ordering that interleaves families before adding variants. This ensures that the five-agent sample is not accidentally composed of one specialist family. The experiment therefore increases both headcount and intellectual coverage in a controlled way.

The output is a compact convergence table and two charts: holdout RMSE against swarm size and the number of selected features against swarm size. Together they show whether collective intelligence is producing a more accurate and stable equation or merely a more complicated one.

In [19]:
# CELL 9 — Swarm-size convergence: do 50 agents add information?
# Interleave families so small subswarms have broad intellectual coverage.
ordered = contracts.sort_values(["variant","family"]).reset_index(drop=True)

def features_from_subswarm(sub):
    rr = []
    for _, row in sub.iterrows():
        for f in row["features"]:
            rr.append((row["family"], f))
    vv = pd.DataFrame(rr, columns=["family","feature"])
    ss = vv.groupby("feature").agg(agent_support=("feature","size"), family_support=("family","nunique"))
    ss = ss.sort_values(["family_support","agent_support"], ascending=False)
    k = min(MAX_FINAL_FEATURES, max(6, int(round(np.sqrt(len(sub))*2))))
    return ss.head(k).index.tolist()

def fit_eval_with_features(features):
    local_models = {}
    for typ in ["put","call"]:
        d = train[train.type==typ].copy()
        if typ=="call":
            d = d[d.q>1e-8]
        Z = feature_frame(d)[features]
        X = np.column_stack([np.ones(len(Z)),Z.values])
        m = Ridge(alpha=RIDGE_ALPHA,fit_intercept=False).fit(X,d.premium_norm.values)
        local_models[typ]=m

    def local_predict(d):
        Z = feature_frame(d)[features]
        ans=[]
        for pos, (_, row) in enumerate(d.iterrows()):
            typ=row.type; euro=row.euro
            intrinsic=max(row.S-row.K,0) if typ=="call" else max(row.K-row.S,0)
            if typ=="call" and row.q<=1e-10:
                p=euro
            else:
                raw=local_models[typ].predict(np.r_[1.0,Z.iloc[pos].values].reshape(1,-1))[0]
                p=euro+row.K*max(0,raw)
            upper=row.S if typ=="call" else row.K
            ans.append(min(max(p,euro,intrinsic),upper))
        return np.array(ans)

    pred=local_predict(test)
    rmse=np.sqrt(np.mean((pred-test.american.values)**2))
    mae=np.mean(np.abs(pred-test.american.values))
    return rmse,mae

conv=[]
for n in [5,10,20,30,40,50]:
    n=min(n,len(ordered))
    feats=features_from_subswarm(ordered.head(n))
    rmse,mae=fit_eval_with_features(feats)
    conv.append({"agents":n,"features":len(feats),"RMSE":rmse,"MAE":mae,"basis":", ".join(feats)})
conv=pd.DataFrame(conv).drop_duplicates("agents")
display(conv)

plt.figure(figsize=(7,4))
plt.plot(conv.agents,conv.RMSE,marker="o")
plt.xlabel("Validated agents in subswarm")
plt.ylabel("Holdout RMSE")
plt.title("Does a larger swarm improve the approximation?")
plt.grid(alpha=.25)
plt.show()

plt.figure(figsize=(7,4))
plt.plot(conv.agents,conv.features,marker="o")
plt.xlabel("Validated agents in subswarm")
plt.ylabel("Selected feature count")
plt.title("Structural complexity versus swarm size")
plt.grid(alpha=.25)
plt.show()

,agents,features,RMSE,MAE,basis
0,5,6,0.297652,0.161539,"call_itm, put_itm, varT, carryT, sig_sqrtT, di..."
1,10,6,0.245072,0.153471,"call_itm, put_itm, varT, carryT, sig_sqrtT, x_qT"
2,20,9,0.187537,0.104987,"call_itm, put_itm, varT, sig_sqrtT, carryT, di..."
3,30,11,0.187997,0.104956,"call_itm, put_itm, varT, sig_sqrtT, qT, rT, ca..."
4,40,13,0.191408,0.105571,"call_itm, put_itm, varT, sig_sqrtT, carryT, qT..."
5,50,14,0.191275,0.104729,"call_itm, put_itm, varT, sig_sqrtT, carryT, qT..."


<Figure size 700x400 with 1 Axes>

<Figure size 700x400 with 1 Axes>

## CELL 10 — Final equation, stress test, reproducibility report, and research closure

The final unit consolidates the experiment into a research artifact. It prints the final put and dividend-paying-call equations, the exact non-dividend-paying-call rule, the feature definitions, the training domain, and the principal holdout metrics. This creates a self-contained record that can be copied into a paper, lecture, or downstream pricing prototype.

The cell then performs an out-of-sample stress test on a deliberately harder parameter region. The stress sample extends toward shorter maturities, higher volatility, more extreme moneyness, and larger dividend yields than the central training distribution. The objective is not to claim universal robustness. It is to expose how quickly the approximation deteriorates as it leaves the region in which it was estimated. The stress errors are therefore interpreted as scope diagnostics.

A reproducibility report records the model name, number of successful agents, selected features, tree depth, ridge penalty, random seed, and timestamp. The final swarm provenance table is saved to CSV together with the fitted coefficient table and validation metrics. This makes the notebook more than a visual demonstration: it leaves machine-readable evidence behind.

The methodological closure is equally important. The notebook distinguishes three layers of knowledge. Option theory contributes invariants and decomposition. The agent swarm contributes hypotheses about compact structure. Numerical methods decide coefficients and accuracy. None of those layers substitutes for the others. The final approximate equation is valuable precisely because its origin is inspectable and because the workflow includes explicit mechanisms for disagreement, falsification, and failure.

In [20]:
# CELL 10 — Final equation, stress test, reproducibility report, and research closure
def coef_table():
    rows=[]
    for typ in ["put","call"]:
        coef=models[typ].coef_
        rows.append({"option_type":typ,"term":"INTERCEPT","coefficient":coef[0]})
        rows += [{"option_type":typ,"term":f,"coefficient":c} for f,c in zip(selected_features,coef[1:])]
    return pd.DataFrame(rows)

coefs=coef_table()
display(coefs)

print("\nFINAL APPROXIMATION")
print("Put: P_A = projection_bounds[P_BS + K·max(0, beta_put' phi)]")
print("Dividend call: C_A = projection_bounds[C_BS + K·max(0, beta_call' phi)]")
print("Non-dividend call: C_A = C_BS")
print("\nphi =", ["1"]+selected_features)
print("\nPut linear form:", equation_string("put"))
print("Call linear form:", equation_string("call"))

stress=make_universe(n=500,seed=SEED+99,stress=True)
stress["euro"]=[bs_price(*row) for row in stress[["S","K","T","r","q","sigma","type"]].itertuples(index=False,name=None)]
stress["american"]=[american_crr(*row) for row in stress[["S","K","T","r","q","sigma","type"]].itertuples(index=False,name=None)]
stress["pred"]=predict_american(stress)
stress["abs_err"]=np.abs(stress.pred-stress.american)
stress["sq_err"]=(stress.pred-stress.american)**2

stress_metrics={
    "N":len(stress),
    "MAE":stress.abs_err.mean(),
    "RMSE":np.sqrt(stress.sq_err.mean()),
    "MedianAE":stress.abs_err.median(),
    "P95_AE":stress.abs_err.quantile(.95),
}
print("\nExtended-domain stress test:")
print(pd.Series(stress_metrics))

report={
    "timestamp_utc":datetime.now(timezone.utc).isoformat(),
    "model":MODEL,
    "successful_agents":len(contracts),
    "requested_agents":N_AGENTS,
    "families":N_FAMILIES,
    "tree_steps":TREE_STEPS,
    "ridge_alpha":RIDGE_ALPHA,
    "seed":SEED,
    "selected_features":selected_features,
    "holdout_metrics":summary_metrics.loc["ALL"].to_dict(),
    "stress_metrics":stress_metrics,
}

contracts.to_csv("american_option_swarm_provenance.csv",index=False)
coefs.to_csv("american_option_closed_form_coefficients.csv",index=False)
summary_metrics.to_csv("american_option_holdout_metrics.csv")
with open("american_option_reproducibility_report.json","w") as f:
    json.dump(report,f,indent=2,default=float)

print("\nReproducibility report:")
print(json.dumps(report,indent=2,default=float))
print("\nSaved: provenance CSV, coefficient CSV, holdout metrics CSV, reproducibility JSON.")

,option_type,term,coefficient
0,put,INTERCEPT,0.629106
1,put,call_itm,0.000915
2,put,put_itm,0.009212
3,put,varT,-0.001553
4,put,sig_sqrtT,-0.001024
5,put,carryT,0.123673
6,put,qT,-0.277019
7,put,rT,-0.153346
8,put,x_rT,0.518102
9,put,disc_q,-0.438950



FINAL APPROXIMATION
Put: P_A = projection_bounds[P_BS + K·max(0, beta_put' phi)]
Dividend call: C_A = projection_bounds[C_BS + K·max(0, beta_call' phi)]
Non-dividend call: C_A = C_BS

phi = ['1', 'call_itm', 'put_itm', 'varT', 'sig_sqrtT', 'carryT', 'qT', 'rT', 'x_rT', 'disc_q', 'disc_r', 'x_qT', 'sqrtT_q', 'sqrtT_r', 'call_itm2']

Put linear form: +0.62910621 +0.00091456021*call_itm +0.0092116106*put_itm -0.0015528651*varT -0.0010236153*sig_sqrtT +0.12367281*carryT -0.27701918*qT -0.15334637*rT +0.51810165*x_rT -0.43895037*disc_q -0.18999585*disc_r -0.281887*x_qT -0.097047118*sqrtT_q -0.013540764*sqrtT_r +0.0095093008*call_itm2
Call linear form: +0.90221663 +0.0077113913*call_itm +0.006049391*put_itm +0.0064287521*varT -0.0075884689*sig_sqrtT -0.043106706*carryT -0.32524121*qT -0.36834791*rT +0.24228089*x_rT -0.43378504*disc_q -0.46831902*disc_r -0.63351367*x_qT +0.026812647*sqrtT_q -0.10788213*sqrtT_r +0.016513749*call_itm2

Extended-domain stress test:
N           500.000000
MAE   

## Investor Memorandum: American Option Approximation Formula

**To:** Interested Investors
**From:** Quantitative Research Team
**Date:** (Current Date)
**Subject:** Derivation and Validation of an Analytical Approximation for American Option Pricing

#### 1. Executive Summary

This memorandum outlines the development and validation of a new analytical approximation for pricing American options, specifically focusing on its `closed-form` nature for practical application. Leveraging a collective intelligence framework (fifty `Claude Sonnet 5` agents), we have derived a robust equation that significantly streamlines the valuation process for these complex derivatives. The approximation covers put options, dividend-paying call options, and non-dividend-paying call options, offering a balance between accuracy and computational efficiency.

#### 2. The Final Approximation Formula

The core of our research is a set of empirically derived equations that approximate the early exercise premium for American options. The final formulas are structured as follows:

*   **American Put Option (P_A):**
    `P_A = projection_bounds[P_BS + K ⋅ max(0, β_put' φ)]`

*   **Dividend-Paying American Call Option (C_A):**
    `C_A = projection_bounds[C_BS + K ⋅ max(0, β_call' φ)]`

*   **Non-Dividend-Paying American Call Option (C_A):**
    `C_A = C_BS` (The American value equals the European value, as per established theory)

Where:
*   `P_BS` and `C_BS` are the Black-Scholes prices for European put and call options, respectively.
*   `K` is the strike price.
*   `max(0, ...)` ensures the early exercise premium is non-negative.
*   `β_put'` and `β_call'` represent vectors of coefficients for put and dividend-paying call options, respectively, determined by Ridge regression.
*   `φ` is the vector of selected dimensionless features (mathematical transformations of underlying option parameters) that drive the early exercise premium. These features include variations of moneyness, volatility-time interactions, carry costs, and interest rate effects. The specific features used are: `['1', 'call_itm', 'put_itm', 'varT', 'sig_sqrtT', 'carryT', 'qT', 'rT', 'x_rT', 'disc_q', 'disc_r', 'x_qT', 'sqrtT_q', 'sqrtT_r', 'call_itm2']`.
*   `projection_bounds[...]` ensures the final price respects no-arbitrage conditions (i.e., it is greater than intrinsic value and European value, and within simple upper bounds).

**Linear Forms for Premium (for illustration, coefficients will vary with model retraining):**
*   **Put:** `+0.62910621 +0.00091456021*call_itm +0.0092116106*put_itm -0.0015528651*varT -0.0010236153*sig_sqrtT +0.12367281*carryT -0.27701918*qT -0.15334637*rT +0.51810165*x_rT -0.43895037*disc_q -0.18999585*disc_r -0.281887*x_qT -0.097047118*sqrtT_q -0.013540764*sqrtT_r +0.0095093008*call_itm2`
*   **Dividend Call:** `+0.90221663 +0.0077113913*call_itm +0.006049391*put_itm +0.0064287521*varT -0.0075884689*sig_sqrtT -0.043106706*carryT -0.32524121*qT -0.36834791*rT +0.24228089*x_rT -0.43378504*disc_q -0.46831902*disc_r -0.63351367*x_qT +0.026812647*sqrtT_q -0.10788213*sqrtT_r +0.016513749*call_itm2`

These linear forms represent the `β' φ` component, where `β` are the fitted coefficients for each feature.

#### 3. Extended-Domain Stress Test Results

To assess the model's robustness beyond its primary training distribution, an extended-domain stress test was conducted on 500 options with parameters pushing towards shorter maturities, higher volatility, more extreme moneyness, and larger dividend yields. The results are as follows:

*   **N**: 500
*   **MAE (Mean Absolute Error)**: 0.2355
*   **RMSE (Root Mean Squared Error)**: 0.4970
*   **MedianAE (Median Absolute Error)**: 0.1090
*   **P95_AE (95th Percentile Absolute Error)**: 0.8146

While these errors are higher than those observed in the core holdout validation set (where MAE was ~0.105), this is an expected outcome. The stress test demonstrates that the approximation's accuracy naturally deteriorates in regions significantly outside its training domain. This provides crucial insight into the model's applicability boundaries, guiding where further model refinement or alternative valuation methods might be necessary.

#### 4. Reproducibility and Auditability

Our methodology emphasizes transparency and reproducibility. A comprehensive reproducibility report is generated, capturing all critical model parameters and performance metrics:

*   **Timestamp (UTC):** The exact time of model generation.
*   **Model:** `claude-sonnet-5` (referring to the AI agents).
*   **Successful Agents:** 50 (all agents provided valid contributions).
*   **Requested Agents:** 50.
*   **Families:** 10 (distinct intellectual families of agents).
*   **Tree Steps:** 300 (for the Cox-Ross-Rubinstein benchmark).
*   **Ridge Alpha:** 1e-05 (regularization parameter).
*   **Seed:** 766 (for random number generation, ensuring replicability).
*   **Selected Features:** The list of 14 features ultimately chosen for the final approximation.
*   **Holdout Metrics:** Key performance indicators from the initial validation (e.g., MAE: 0.1047, RMSE: 0.1913).
*   **Stress Metrics:** Performance indicators from the extended-domain stress test.

All provenance data, including agent contributions, fitted coefficients, holdout metrics, and this reproducibility report, are saved to machine-readable files (`CSV` and `JSON`). This ensures full auditability and enables future research to build upon or challenge these findings in a verifiable manner.

#### 5. Conclusion for Investors

This analytical approximation provides a powerful tool for rapidly valuing American options, particularly in environments where computational speed is paramount. Its 'closed-form' nature, once coefficients are determined, eliminates the need for time-consuming tree-based or iterative numerical methods at inference. The rigorous validation, including stress testing and transparent reproducibility, offers confidence in its application within its defined scope. Investors should understand that while highly accurate within typical market conditions, caution is advised when pricing options in extreme, out-of-sample scenarios, as indicated by the stress test results. This framework represents a significant step towards leveraging advanced AI for robust and interpretable quantitative finance solutions.


## Conclusion

This notebook has treated the search for an approximate American-option equation as a governed collective-research problem rather than as a prompt-engineering stunt. Fifty Claude Sonnet 5 agents were organized into ten specialist families, each family contributing five differentiated variants. Their role was deliberately narrow but intellectually meaningful: propose compact mathematical structure for the early-exercise premium, explain why the proposed variables matter, and identify likely failure modes. They were not allowed to invent prices, estimate their own accuracy, or silently modify the benchmark.

The numerical layer remained independent. European Black–Scholes values supplied the analytic anchor, while a Cox–Ross–Rubinstein tree supplied American-option reference prices. This allowed the central object to be written as a European price plus an early-exercise premium. The swarm then voted over a controlled library of dimensionless state variables. Validation converted free-form model reasoning into governed contracts; family-level support reduced the danger of treating correlated variants as independent discoveries; deterministic regression estimated coefficients; and explicit projections enforced elementary pricing bounds.

The resulting expression is therefore “closed form” in an operational, approximate sense: once the coefficients have been fitted, evaluation requires only the Black–Scholes formula, elementary transformations of $S,K,T,r,q,\sigma$, a finite weighted sum, and a positive-part/bound projection. No tree or iterative free-boundary solver is needed at inference time. For a non-dividend-paying call, the notebook does not approximate what theory already gives exactly: it sets the American value equal to the European call value.

The holdout and regime diagnostics are as important as the equation itself. A compact formula should be judged not only by average error but by where it fails. Short maturity, extreme moneyness, high volatility, and strong dividend incentives are precisely the regions in which a smooth global approximation may struggle. The worst-case table and extended-domain stress test expose those weaknesses. They should be read as invitations to refine the representation—for example, by adding regime-specific terms or a learned exercise-boundary proxy—not as inconveniences to be hidden.

The subswarm experiment addresses a second research question. If the basis and validation error stabilize before all fifty agents are included, the exercise provides evidence that additional agents are becoming redundant under the current architecture. If new families continue to introduce useful terms and error continues to fall, the larger swarm has measurable informational value. In either case, swarm size becomes an empirical design variable rather than a slogan.

Several extensions are natural. One could compare the swarm-derived expression with established analytical approximations such as Barone-Adesi–Whaley or Bjerksund–Stensland, replace the CRR benchmark with a high-resolution finite-difference solver, introduce discrete dividends, or allow the swarm to propose a small number of regime-specific equations rather than one global premium function. A more ambitious version could ask one subswarm to propose structures, a second to attack them, and a third to design falsification tests, while preserving the deterministic pricing layer as final arbiter.

The broader lesson extends beyond derivatives. Advanced autonomous systems are most useful when generative and deterministic intelligence are assigned different institutional responsibilities. The agents create hypotheses and expose alternative representations. Numerical methods impose measurement. Governance validates contracts and preserves provenance. Stress testing searches for failure. The final result is not “the opinion of fifty agents”; it is a compact mathematical artifact that has survived a transparent process of collective proposal and independent numerical adjudication.

That is the central purpose of this notebook: not to claim that a swarm has solved the American-option free-boundary problem exactly, but to demonstrate how a large, heterogeneous AI research collective can be used to **discover, compress, test, and falsify an interpretable approximation**. The equation is the visible output. The more important contribution is the architecture that produced it.